# AutoPBR / MASON pipeline — Run notebook

Questo notebook orchestri i tuoi step Python in ordine, mantenendo **gli stessi comandi CLI** che useresti da terminale.
È pensato per essere copiato nel repo come `run_pipeline.ipynb`.

> Nota: se preferisci, puoi spostare gli script sotto `scripts/` e aggiornare la variabile `SCRIPTS_DIR`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shlex, json

# =========================
# CONFIG (edit these)
# =========================
REPO_ROOT = Path(".").resolve()
SCRIPTS_DIR = REPO_ROOT  # oppure REPO_ROOT / "scripts"

# Input/output (coerenti con i tuoi script)
DATA_INPUT  = REPO_ROOT / "data_input"
DATA_OUTPUT = REPO_ROOT / "data_output"

# Env vars (impostale in locale, oppure qui in modo temporaneo)
# os.environ["MAPILLARY_TOKEN"] = "MLY|..."
# os.environ["NVIDIA_API_KEY"]  = "nvapi-..."

print("Repo:", REPO_ROOT)
print("Scripts:", SCRIPTS_DIR)


## Helper per eseguire comandi (con output live)

Usiamo `subprocess.run` così hai la stessa esperienza del terminale (utile anche per debug e reproducibility).


In [ ]:
def run(cmd: str):
    print("▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def py(script: str, args: str = ""):
    script_path = (SCRIPTS_DIR / script).resolve()
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")
    run(f"{sys.executable} {script_path} {args}".strip())


## Step 0 — Sanity checks

- Python e cartelle
- Env vars richieste dagli step con API


In [ ]:
DATA_INPUT.mkdir(exist_ok=True, parents=True)
DATA_OUTPUT.mkdir(exist_ok=True, parents=True)

missing = []
if not os.environ.get("MAPILLARY_TOKEN"):
    missing.append("MAPILLARY_TOKEN (Mapillary)")
if not os.environ.get("NVIDIA_API_KEY"):
    missing.append("NVIDIA_API_KEY (Nemotron via NVIDIA integrate API)")

if missing:
    print("⚠️ Env vars mancanti:", ", ".join(missing))
    print("   Puoi settarle nel terminale prima di lanciare Jupyter, oppure in alto nel notebook.")
else:
    print("✅ Env vars OK")


## Step 1 — Download data (OSM + buildings geojson + Mapillary photos + manifest)

Lo script salva tipicamente in `data_input/` e genera `photos_manifest.json`.


In [ ]:
# Esempio bbox (sostituisci con la tua)
BBOX = "5.32045,60.39670,5.32650,60.39990"

# Nota: nello script lo step è pensato tipo:
#   python 1_download_data.py --bbox "..." --profile autopbr --token ...
# Se non passi --token, usa MAPILLARY_TOKEN da env.
py("1_download_data.py", f'--bbox "{BBOX}" --profile autopbr')


## Step 2 — SAM3 structure-first masks (road / wall / roof + building instances)

Questo step produce un manifest in `data_output/sam3_instances/manifest.json` con:
- maschere globali road/wall/roof
- maschere per-building (instances)


In [ ]:
# Cambia pattern se le immagini sono .jpg
IMAGES_DIR = DATA_INPUT / "photos"  # oppure data_input/images
OUT_DIR = DATA_OUTPUT / "sam3_instances"

py("2_sam3hi.py", f'--images-dir "{IMAGES_DIR}" --out-dir "{OUT_DIR}" --pattern "*.jpg"')


## Step 3 — Proxy semantic sanity check (ADE SegFormer + OneFormer)

Confronta le maschere SAM3 con due proxy semantic segmentation (road/wall/roof super-classes).
Utile come sanity check (non è GT).


In [ ]:
MANIFEST = DATA_OUTPUT / "sam3_instances" / "manifest.json"
PROXY_OUT = DATA_OUTPUT / "proxy_check"

# Aggiungi --save_overlays se vuoi PNG di debug
py("4_proxy_semantic_check.py", f'--manifest "{MANIFEST}" --out_dir "{PROXY_OUT}" --save_overlays')


## Step 4 — Nemotron per-region (maschere cropped + descrittori materiali)

Questo è lo step "structure-first → material descriptor" con immagini mascherate/croppate.
Produce un JSON tipo `data_output/materials_from_nemotron.json`.


In [ ]:
MASKED_DIR = DATA_OUTPUT / "masked_rgb"
OUT_JSON = DATA_OUTPUT / "materials_from_nemotron.json"

# Puoi attivare per-building con --per_building (se vuoi wall/roof per edificio)
# e filtri come --skip_far_buildings.
py("4_nemotron_per_imagev2_UPDATED.py",
   f'--manifest "{MANIFEST}" --masked_dir "{MASKED_DIR}" --out_json "{OUT_JSON}" --per_building --skip_far_buildings')


## Step 5 (opzionale) — Nemotron full-image baseline

Baseline alternativa: material recognition su immagine intera (meno robusta, ma utile come confronto).


In [ ]:
# Se vuoi eseguire anche il baseline full-image, scommenta:
# py("5_nemotron_fullimage.py", f'--input_dir "{IMAGES_DIR}" --out_json "{DATA_OUTPUT / "materials_fullimage_baseline.json"}"')
print("Step 5 è opzionale. Scommenta la cella se ti serve.")


## Output attesi (quick view)


In [ ]:
print("📁 data_input:", list(DATA_INPUT.glob("*"))[:10])
print("📁 data_output:", list(DATA_OUTPUT.glob("*"))[:10])

paths = [
    DATA_INPUT / "photos_manifest.json",
    MANIFEST,
    DATA_OUTPUT / "materials_from_nemotron.json",
    PROXY_OUT / "proxy_global_iou.csv",
]
for p in paths:
    print(("✅" if p.exists() else "❌"), p)
